<a href="https://colab.research.google.com/github/hypher777/tiny-gpt-from-scratch/blob/main/tiny_gpt_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:

import sys

print("Python version:", sys.version)

try:
    import torch
    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("We will start with CPU.")
except ImportError:
    print("PyTorch is not installed yet.")


Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch version: 2.11.0+cpu
CUDA available: False
We will start with CPU.


In [36]:
text = """
The strong survive through preparation.
A calm mind sees hidden opportunities.
Power means nothing without resolve.
Never mistake patience for weakness.
The truth is not always what it seems.
Protect those who matter to you.
Sacrifice can carry a heavy price.
A shinobi must see beyond appearances.
Ambition demands difficult choices.
Even darkness can teach a lesson.
Trust must be earned through actions.
The future belongs to those who prepare.
"""

print("training text :")
print(text)
print("number of text :")
print(len(text))
characters=sorted(set(text))
print("\n unique characters :")
print(characters)
print("\n number of unique characters :")
print(len(characters))

training text :

The strong survive through preparation.
A calm mind sees hidden opportunities.
Power means nothing without resolve.
Never mistake patience for weakness.
The truth is not always what it seems.
Protect those who matter to you.
Sacrifice can carry a heavy price.
A shinobi must see beyond appearances.
Ambition demands difficult choices.
Even darkness can teach a lesson.
Trust must be earned through actions.
The future belongs to those who prepare.

number of text :
449

 unique characters :
['\n', ' ', '.', 'A', 'E', 'N', 'P', 'S', 'T', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y']

 number of unique characters :
31


In [37]:

# Our tiny training text
text = """
The strong survive through preparation.
A calm mind sees hidden opportunities.
Power means nothing without resolve.
Never mistake patience for weakness.
The truth is not always what it seems.
Protect those who matter to you.
Sacrifice can carry a heavy price.
A shinobi must see beyond appearances.
Ambition demands difficult choices.
Even darkness can teach a lesson.
Trust must be earned through actions.
The future belongs to those who prepare.
"""


# Collect every unique character and sort them
characters = sorted(set(text))

# Give each character a unique number
stoi = {}
for index, character in enumerate(characters):
    stoi[character] = index

# Convert numbers back into characters
itos = {}
for character, index in stoi.items():
    itos[index] = character


# Convert text into a list of numbers
def encode(sentence):
    token_ids = []

    for character in sentence:
        token_ids.append(stoi[character])

    return token_ids


# Convert a list of numbers back into text
def decode(token_ids):
    sentence = ""

    for index in token_ids:
        sentence += itos[index]

    return sentence


# Test both directions
encoded = encode("hello")
decoded = decode(encoded)

print("Encoded:", encoded)
print("Decoded:", decoded)
print("Correct:", decoded == "hello")


Encoded: [16, 13, 19, 19, 22]
Decoded: hello
Correct: True


In [38]:

import torch

# Convert every character into its number.
# PyTorch tensors are the data format our model will use.
data = torch.tensor(encode(text), dtype=torch.long)

# Check how many character tokens we have.
print("Total tokens:", len(data))


Total tokens: 449


In [39]:

# How many characters the model can look at at once.
block_size = 8

# Input: the first 8 characters.
x = data[:block_size]

# Target: the same sequence shifted forward by one character.
y = data[1:block_size + 1]

# Decode the numbers so we can understand the example.
print("Input: ", repr(decode(x.tolist())))
print("Target:", repr(decode(y.tolist())))


Input:  '\nThe str'
Target: 'The stro'


In [40]:

# How many training examples we process together
batch_size = 4

# Number of characters the model can look at
# This should already be 8 from our previous module.
block_size = 8

# Pick random starting positions in the training text.
# Leave enough room for the input and its next-character targets.
starts = torch.randint(
    len(data) - block_size,
    (batch_size,)
)

# Store the input sequences and their targets.
x_batch = []
y_batch = []

for start in starts:
    # Convert the random position into a normal Python integer.
    start = start.item()

    # Input: a sequence of characters from this position.
    x = data[start : start + block_size]

    # Target: the same sequence shifted forward by one character.
    y = data[start + 1 : start + block_size + 1]

    x_batch.append(x)
    y_batch.append(y)

# Combine the individual sequences into two tensors.
x_batch = torch.stack(x_batch)
y_batch = torch.stack(y_batch)

print("Input batch shape:", x_batch.shape)
print("Target batch shape:", y_batch.shape)


Input batch shape: torch.Size([4, 8])
Target batch shape: torch.Size([4, 8])


In [41]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Number of unique characters in our training text
vocab_size = len(characters)

# A tiny model: each input character gets scores for every possible next character
model = nn.Embedding(vocab_size, vocab_size)

print("Vocabulary size:", vocab_size)
print(model)



Vocabulary size: 31
Embedding(31, 31)


In [42]:
# Give the model the first character in our training text
input_character = data[0]

# Calculate scores for every possible next character
logits = model(input_character)

print("Input character:", decode([input_character.item()]))
print("Scores shape:", logits.shape)


Input character: 

Scores shape: torch.Size([31])


In [43]:
# Every character except the last is an input
inputs = data[:-1]

# Every character except the first is its corresponding target
targets = data[1:]

# Predict the next character for every input
logits = model(inputs)

# Measure how far the predictions are from the correct targets
loss = F.cross_entropy(logits, targets)

print("Input shape:", inputs.shape)
print("Target shape:", targets.shape)
print("Initial loss:", loss.item())

Input shape: torch.Size([448])
Target shape: torch.Size([448])
Initial loss: 4.009060382843018


In [44]:

# Create an optimizer connected to our current model.
# If you recreate the model, you must recreate the optimizer too.
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

print("Optimizer ready!")

Optimizer ready!


In [45]:
# Train the model repeatedly
for step in range(100):

    # 1. Clear gradients from the previous step
    optimizer.zero_grad()

    # 2. Predict the next character
    logits = model(inputs)

    # 3. Measure prediction error
    loss = F.cross_entropy(logits, targets)

    # 4. Calculate gradients
    loss.backward()

    # 5. Update the model's parameters
    optimizer.step()

    # Print progress every 20 steps
    if (step + 1) % 20 == 0:
        print(f"Step {step + 1}: Loss = {loss.item():.4f}")

Step 20: Loss = 3.7238
Step 40: Loss = 3.4522
Step 60: Loss = 3.2145
Step 80: Loss = 3.0104
Step 100: Loss = 2.8371


In [46]:
# Start generation with a single character
current = torch.tensor([stoi["T"]], dtype=torch.long)

# Generate 100 characters
generated = []

with torch.no_grad():
    for _ in range(100):
        # Predict the next character
        logits = model(current[-1])

        # Convert scores into probabilities
        probabilities = F.softmax(logits, dim=-1)

        # Choose the next character
        next_character = torch.multinomial(probabilities, num_samples=1)

        # Save it and use it for the next prediction
        generated.append(next_character.item())
        current = torch.cat([current, next_character])

print(decode(generated))

. mAkgskeT.wngy
uwtStnd
peTnfokghivcy
AcaesowedpT k.ton hen
PgwyovicyPr hk
ThuS
ANu otitghalfons.bcv


In [47]:
# Check that the variables needed for our model exist.
print("Text tokenizer exists:", "stoi" in globals())
print("Training data exists:", "data" in globals())
print("Model exists:", "model" in globals())
print("PyTorch loaded:", "torch" in globals())
print("Neural network functions loaded:", "F" in globals())

Text tokenizer exists: True
Training data exists: True
Model exists: True
PyTorch loaded: True
Neural network functions loaded: True


In [48]:

# Each training example will use 8 characters of context.
block_size = 8

# X stores the input contexts.
# Y stores the correct next character for each context.
X = []
Y = []

# Slide through the text one character at a time.
for i in range(len(data) - block_size):
    context = data[i : i + block_size]
    target = data[i + block_size]

    X.append(context)
    Y.append(target)

# Convert the lists into PyTorch tensors.
X = torch.stack(X)
Y = torch.stack(Y)

# Inspect the resulting training data.
print("Input shape:", X.shape)
print("Target shape:", Y.shape)
print("First context:", decode(X[0].tolist()))
print("Correct next character:", decode([Y[0].item()]))


Input shape: torch.Size([441, 8])
Target shape: torch.Size([441])
First context: 
The str
Correct next character: o


In [49]:

import torch.nn as nn
import torch.nn.functional as F

# Number of unique characters in our text.
vocab_size = len(stoi)

# A small neural network that uses the full context.
class ContextModel(nn.Module):
    def __init__(self, vocab_size, block_size):
        super().__init__()

        # Convert each character ID into a vector of 16 numbers.
        self.embedding = nn.Embedding(vocab_size, 16)

        # Combine the embeddings of all 8 characters.
        self.linear1 = nn.Linear(block_size * 16, 128)

        # Produce one prediction score for every possible next character.
        self.linear2 = nn.Linear(128, vocab_size)

    def forward(self, x):
        # x shape: (batch_size, 8)
        x = self.embedding(x)

        # Flatten 8 character vectors into one vector.
        x = x.reshape(x.shape[0], -1)

        # Learn patterns using a hidden layer.
        x = torch.relu(self.linear1(x))

        # Return scores for all possible next characters.
        logits = self.linear2(x)
        return logits


# Create the new model.
model = ContextModel(vocab_size, block_size)

# Check its prediction output shape using one example.
sample = X[:1]
output = model(sample)

print("Input shape:", sample.shape)
print("Output shape:", output.shape)
print("Vocabulary size:", vocab_size)


Input shape: torch.Size([1, 8])
Output shape: torch.Size([1, 31])
Vocabulary size: 31


In [50]:

# Create an optimizer for the current model's parameters.
# Adam adjusts the parameters using gradients calculated during training.
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Train for 500 iterations.
for step in range(500):

    # 1. Clear gradients left over from the previous iteration.
    optimizer.zero_grad()

    # 2. Forward pass: predict the next character for every example.
    logits = model(X)

    # 3. Compare predictions with the correct next characters.
    loss = F.cross_entropy(logits, Y)

    # 4. Backpropagation: calculate gradients for model parameters.
    loss.backward()

    # 5. Update the model parameters using those gradients.
    optimizer.step()

    # Print the loss every 50 iterations.
    if (step + 1) % 50 == 0:
        print(f"Step {step + 1}: Loss = {loss.item():.4f}")


Step 50: Loss = 1.6579
Step 100: Loss = 0.3825
Step 150: Loss = 0.0885
Step 200: Loss = 0.0419
Step 250: Loss = 0.0269
Step 300: Loss = 0.0199
Step 350: Loss = 0.0161
Step 400: Loss = 0.0137
Step 450: Loss = 0.0121
Step 500: Loss = 0.0110


In [51]:

# Switch the model into evaluation mode.
model.eval()

# Take the first training example.
context = X[0:1]

# Disable gradient calculations because we are only testing.
with torch.no_grad():
    logits = model(context)

    # Find the character with the highest prediction score.
    predicted_id = logits.argmax(dim=1).item()

# Retrieve the correct target character.
actual_id = Y[0].item()

print("Input context:", decode(context[0].tolist()))
print("Predicted next character:", decode([predicted_id]))
print("Actual next character:", decode([actual_id]))
print("Prediction correct:", predicted_id == actual_id)


Input context: 
The str
Predicted next character: o
Actual next character: o
Prediction correct: True


In [52]:

# Generate text using the trained context-aware model.

model.eval()

# Start with the first 8 characters from the training text.
current = X[0].clone()

# Store the generated characters separately.
generated = current.tolist()

# Generate 200 new characters.
with torch.no_grad():
    for _ in range(200):

        # Add a batch dimension: (8,) becomes (1, 8).
        context = current[-block_size:].unsqueeze(0)

        # Predict scores for the next character.
        logits = model(context)

        # Get the scores for this single example.
        logits = logits[0]

        # Convert scores into probabilities.
        probabilities = F.softmax(logits, dim=-1)

        # Sample the next character from those probabilities.
        next_character = torch.multinomial(
            probabilities, num_samples=1
        ).item()

        # Add the prediction to our generated sequence.
        generated.append(next_character)

        # Keep the latest 8 characters for the next prediction.
        current = torch.cat([
            current,
            torch.tensor([next_character], dtype=torch.long)
        ])

# Print only the newly generated text.
print(decode(generated[block_size:]))


ong survive through preparation.
A calm mind sees hidden opportunities.
Power means nothing without resolve.
Never mistake patience for weakness.
The truth is not always what it seems.
Protect those w
